In [2]:
import torch
import testdata
import pandas as pd
from multicor_fa import mcfa_model

In [3]:
%load_ext autoreload

In [4]:
%autoreload 2

Intersim data

In [5]:
cluster_df = pd.read_csv(
    '../../sim_data/clustering_assignments.tsv', sep='\t', index_col=1
)
cluster_df.head()

,subjects,cluster.id
subject1,1,2
subject2,2,1
subject3,3,5
subject4,4,3
subject5,5,2


In [6]:
exp_data = pd.read_csv('../../sim_data/expression_data.tsv', sep='\t', index_col=0)
methyl_data = pd.read_csv('../../sim_data/methylation_data.tsv', sep='\t', index_col=0)
protein_data = pd.read_csv('../../sim_data/protein_data.tsv', sep='\t', index_col=0)

# Limit to first 1000 samples for now
exp_data = exp_data.iloc[:1000,:]
methyl_data = methyl_data.iloc[:1000,:]
protein_data = protein_data.iloc[:1000,:]
cluster_df = cluster_df.iloc[:1000]

In [7]:
# Create data dictionary
Y = {
    'exp': exp_data, 
    'methyl': methyl_data,
    'prot': protein_data
}

Complete data

In [8]:
for name, df in Y.items():
    print(f"Missing values in {name}:", df.isna().sum().sum())

Missing values in exp: 0
Missing values in methyl: 0
Missing values in prot: 0


In [9]:
print(cluster_df.shape, Y['exp'].shape, Y['methyl'].shape, Y['prot'].shape)

(1000, 2) (1000, 131) (1000, 367) (1000, 160)


In [43]:
%%time
mcfa_res_full = mcfa_model.fit(Y, missing_modes='raise')

Calculating data PCs.
tensor(6)
tensor(6)
tensor(6)
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.
There are 6 components above rho inclusion threshold 1.1948797702789307.
tensor(6)
tensor(6)
tensor(6)
Fitting the model.
iter: 0 Likelihood: 29936.18468582197
Iter: 1 Likelihood: 29166.527101717576 Percent change: 0.02638838629708054 Time (s): 0.0003490447998046875
Iter: 2 Likelihood: 29069.997232214908 Percent change: 0.0033206012622421484 Time (s): 0.000659942626953125
Iter: 3 Likelihood: 29042.178938393467 Percent change: 0.000957858357682172 Time (s): 0.0009639263153076172
Iter: 4 Likelihood: 29031.473985578607 Percent change: 0.00036873611102824805 Time (s): 0.0012612342834472656
Iter: 5 Likelihood: 29026.252612599616 Percent change: 0.00017988450140906268 Time (s): 0.0016169548034667969
Iter: 6 Likelihood: 29022.934987337143 Percent change: 0.00011431046735693933 Time (s): 0.0019731521606445312
Iter: 7 Likelihood: 29020.41068497861 Percen

In [ ]:
# Shapes of W
print(
    mcfa_res_full.W['exp'].shape,
    mcfa_res_full.W['methyl'].shape,
    mcfa_res_full.W['prot'].shape
)

(131, 6) (367, 6) (160, 6)


In [46]:
# Shapes of L
print(
    mcfa_res_full.L['exp'].shape,
    mcfa_res_full.L['methyl'].shape,
    mcfa_res_full.L['prot'].shape
)

(131, 0) (367, 0) (160, 0)


In [ ]:
# Shapes of Phi
print(
    mcfa_res_full.Phi['exp'].shape,
    mcfa_res_full.Phi['methyl'].shape,
    mcfa_res_full.Phi['prot'].shape
)

(6, 6) (6, 6) (6, 6)


In [24]:
print(
    (mcfa_res_full.W['exp'] @ mcfa_res_full.W['exp'].T).values[:5,:5].round(2)
)

[[ 0.8  -0.06  0.39 -0.21  0.28]
 [-0.06  0.83  0.03 -0.41 -0.38]
 [ 0.39  0.03  0.7  -0.31  0.24]
 [-0.21 -0.41 -0.31  0.65  0.25]
 [ 0.28 -0.38  0.24  0.25  0.55]]


In [22]:
print(
    (mcfa_res_full.L['exp'] @ mcfa_res_full.L['exp'].T).values[:5,:5].round(2)
)

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [25]:
print((mcfa_res_full.Phi['exp']))

          0         1         2         3        4        5
0  0.341021  0.000000  0.000000  0.000000  0.00000  0.00000
1  0.000000  0.362129  0.000000  0.000000  0.00000  0.00000
2  0.000000  0.000000  0.294544  0.000000  0.00000  0.00000
3  0.000000  0.000000  0.000000  0.347802  0.00000  0.00000
4  0.000000  0.000000  0.000000  0.000000  0.36505  0.00000
5  0.000000  0.000000  0.000000  0.000000  0.00000  0.46879


Missing data, with each sample missing maximum 1 mode

In [13]:
Y_miss = Y.copy()
Y_miss['exp'] = Y_miss['exp'].iloc[20:]
Y_miss['methyl'] = Y_miss['methyl'].drop(
    index=Y_miss['methyl'].iloc[100:125].index.tolist()
)
Y_miss['prot'] = Y_miss['prot'].drop(
    index=Y_miss['prot'].iloc[500:530].index.tolist()
)

In [14]:
print(cluster_df.shape, Y_miss['exp'].shape, 
      Y_miss['methyl'].shape, Y_miss['prot'].shape)

(1000, 2) (980, 131) (975, 367) (970, 160)


### Raise error for missing modes

In [15]:
%%time
# Confirm that missing_modes = raise works
mcfa_res_miss_raise = mcfa_model.fit(Y_miss, missing_modes='raise')

ValueError: Missing modes detected for some samples.

### Impute mean

In [16]:
mcfa_res_miss_mean = mcfa_model.fit(Y_miss, missing_modes='impute_mean')

Missing modes detected for some samples, Imputing with the mean
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:300: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:300: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:300: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.1943143606185913.
Fitting the model.
iter: 0 Likelihood: 37635.05531049911
Iter: 1 Likelihood: 36822.02321161364 Percent change: 0.022080049600018803 Time (s): 0.00042700767517089844
Iter: 2 Likelihood: 36735.12380962759 Percent change: 0.0023655671459387522 Time (s): 0.0008039474487304688
Iter: 3 Likelihood: 36713.182126878084 Percent change: 0.0005976513469652135 Time (s): 0.0011169910430908203
Iter: 4 Likelihood: 36705.77378424106 Percent change: 0.00020183044445742552 Time (s): 0.0014128684997558594
Iter: 5 Likelihood: 36702.90223484417 Percent change: 7.823766574400963e-05 Time (s): 0.0017008781433105469
Iter: 6 Likelihood: 36701.603440912986 Percent change: 3.538793429766768e-05 Time (s): 0.0019872188568115234
Iter: 7 Likelihood: 36700.89795472263 Percent change: 1.9222586630520016e-05 Time (s): 0.0022699832916259766
Iter: 8 Likelihood: 36700.4481400229 Percent change: 1.2256381666500399e-05 Time (s): 0.002547025680541992
Ite

In [17]:
print(
    (mcfa_res_miss_mean.W['exp'] @ mcfa_res_miss_mean.W['exp'].T).values[:5,:5].round(2)
)

[[ 0.8  -0.07  0.4  -0.22  0.28]
 [-0.07  0.81  0.05 -0.41 -0.37]
 [ 0.4   0.05  0.69 -0.32  0.24]
 [-0.22 -0.41 -0.32  0.65  0.24]
 [ 0.28 -0.37  0.24  0.24  0.54]]


In [18]:
print((mcfa_res_miss_mean.Phi['exp']))

          0         1         2         3         4         5
0  0.817716  0.000000  0.000000  0.000000  0.000000  0.000000
1  0.000000  0.407358  0.000000  0.000000  0.000000  0.000000
2  0.000000  0.000000  0.552504  0.000000  0.000000  0.000000
3  0.000000  0.000000  0.000000  0.677004  0.000000  0.000000
4  0.000000  0.000000  0.000000  0.000000  0.570731  0.000000
5  0.000000  0.000000  0.000000  0.000000  0.000000  0.556423


### Drop

In [19]:
mcfa_res_miss_drop = mcfa_model.fit(Y_miss, missing_modes='drop')

Missing modes detected for some samples, dropping sampleswith missing modes. There are 925 samples remaining.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.
There are 6 components above rho inclusion threshold 1.2050788402557373.
Fitting the model.
iter: 0 Likelihood: 27700.238499401028
Iter: 1 Likelihood: 26990.502333303924 Percent change: 0.026295774614811494 Time (s): 0.00034689903259277344
Iter: 2 Likelihood: 26902.19184037937 Percent change: 0.0032826504787614166 Time (s): 0.0006656646728515625
Iter: 3 Likelihood: 26876.94375096332 Percent change: 0.0009393958498404734 Time (s): 0.0009768009185791016
Iter: 4 Likelihood: 26867.231164785826 Percent change: 0.0003615030561922899 Time (s): 0.0012798309326171875
Iter: 5 Likelihood: 26862.47084396295 Percent change: 0.00017721083255993505 Time (s): 0.0015759468078613281
Iter: 6 Likelihood: 26859.431948932834 Percent change: 0.00011314070364162826 Time (s): 0.0018696784973

In [20]:
print(
    (mcfa_res_miss_drop.W['exp'] @ mcfa_res_miss_drop.W['exp'].T).values[:5,:5].round(2)
)

[[ 0.81 -0.06  0.4  -0.22  0.28]
 [-0.06  0.83  0.05 -0.4  -0.37]
 [ 0.4   0.05  0.7  -0.33  0.23]
 [-0.22 -0.4  -0.33  0.65  0.24]
 [ 0.28 -0.37  0.23  0.24  0.54]]


In [21]:
print((mcfa_res_miss_drop.Phi['exp']))

          0         1         2         3         4         5
0  0.339415  0.000000  0.000000  0.000000  0.000000  0.000000
1  0.000000  0.367371  0.000000  0.000000  0.000000  0.000000
2  0.000000  0.000000  0.294456  0.000000  0.000000  0.000000
3  0.000000  0.000000  0.000000  0.340908  0.000000  0.000000
4  0.000000  0.000000  0.000000  0.000000  0.367526  0.000000
5  0.000000  0.000000  0.000000  0.000000  0.000000  0.468632


### Impute model

In [63]:
%%time
mcfa_res_miss_impute = mcfa_model.fit(Y_miss, missing_modes='impute_model')

Missing modes detected in input, they will be imputed during model fitting.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:310: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.204781174659729.
Fitting the model.
iter: 0 Likelihood: 38754.08039015749
Iter: 1 Likelihood: 40278.3194267481 Percent change: -0.03784266717886921 Time (s): 0.0003731250762939453
Calculating feature importance.
CPU times: user 1.9 s, sys: 219 ms, total: 2.12 s
Wall time: 377 ms


In [64]:
print(
    (mcfa_res_miss_impute.W['exp'] @ mcfa_res_miss_impute.W['exp'].T).values[:5,:5].round(2)
)

[[ 0.81 -0.07  0.4  -0.22  0.28]
 [-0.07  0.83  0.05 -0.41 -0.38]
 [ 0.4   0.05  0.7  -0.32  0.24]
 [-0.22 -0.41 -0.32  0.65  0.24]
 [ 0.28 -0.38  0.24  0.24  0.55]]


In [65]:
print((mcfa_res_miss_impute.Phi['exp']))

          0         1         2         3         4         5
0  2.665008  0.000000  0.000000  0.000000  0.000000  0.000000
1  0.000000  1.821214  0.000000  0.000000  0.000000  0.000000
2  0.000000  0.000000  1.719293  0.000000  0.000000  0.000000
3  0.000000  0.000000  0.000000  1.441523  0.000000  0.000000
4  0.000000  0.000000  0.000000  0.000000  1.305446  0.000000
5  0.000000  0.000000  0.000000  0.000000  0.000000  1.013158
